# PaddleOCR Compatibility Checker

Tests multiple installation methods to find what works.

In [ ]:
# System Info
import sys
print(f"Python: {sys.version}")
print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

---
## Method 1: Latest PaddleOCR + langchain fix

In [ ]:
# Clean install
!pip uninstall paddlepaddle paddleocr paddlex langchain langchain-community langchain-core -y -q 2>/dev/null

# Install
!pip install paddlepaddle -q
!pip install paddleocr -q
!pip install langchain==0.1.0 langchain-community==0.0.13 -q

print("Method 1 installed. Testing...")

In [ ]:
# Test Method 1
METHOD_1 = False
try:
    import paddle
    print(f"PaddlePaddle: {paddle.__version__}")
    
    from paddleocr import PaddleOCR
    print("Import: OK")
    
    ocr = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)
    print("Init: OK")
    print("\nMETHOD 1: WORKS!")
    METHOD_1 = True
except Exception as e:
    print(f"FAILED: {e}")
    METHOD_1 = False

---
## Method 2: PaddleOCR 2.8.1 + PaddlePaddle 2.6.2

In [ ]:
# Only run if Method 1 failed
if METHOD_1:
    print("Method 1 worked! Skipping Method 2.")
else:
    print("Trying Method 2...")
    !pip uninstall paddlepaddle paddleocr paddlex -y -q 2>/dev/null
    !pip install paddlepaddle==2.6.2 -q
    !pip install paddleocr==2.8.1 -q
    print("Method 2 installed.")

In [ ]:
# Test Method 2
METHOD_2 = False
if not METHOD_1:
    try:
        # Force reload
        import importlib
        import paddle
        importlib.reload(paddle)
        print(f"PaddlePaddle: {paddle.__version__}")
        
        from paddleocr import PaddleOCR
        print("Import: OK")
        
        ocr = PaddleOCR(use_angle_cls=True, lang='en')
        print("Init: OK")
        print("\nMETHOD 2: WORKS!")
        METHOD_2 = True
    except Exception as e:
        print(f"FAILED: {e}")
        METHOD_2 = False

---
## Method 3: Bypass langchain check

In [ ]:
# Only run if Methods 1 and 2 failed
if METHOD_1 or METHOD_2:
    print("Previous method worked! Skipping Method 3.")
else:
    print("Trying Method 3 (bypass check)...")
    !pip uninstall paddlepaddle paddleocr paddlex langchain langchain-community langchain-core -y -q 2>/dev/null
    !pip install paddlepaddle -q
    !pip install paddleocr -q
    print("Method 3 installed.")

In [ ]:
# Test Method 3 - bypass langchain
METHOD_3 = False
if not METHOD_1 and not METHOD_2:
    try:
        import os
        os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'
        
        # Mock langchain to prevent import error
        import sys
        from unittest.mock import MagicMock
        sys.modules['langchain'] = MagicMock()
        sys.modules['langchain.docstore'] = MagicMock()
        sys.modules['langchain.docstore.document'] = MagicMock()
        sys.modules['langchain.text_splitter'] = MagicMock()
        sys.modules['langchain_community'] = MagicMock()
        
        import paddle
        print(f"PaddlePaddle: {paddle.__version__}")
        
        from paddleocr import PaddleOCR
        print("Import: OK")
        
        ocr = PaddleOCR(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)
        print("Init: OK")
        print("\nMETHOD 3: WORKS!")
        METHOD_3 = True
    except Exception as e:
        print(f"FAILED: {e}")
        METHOD_3 = False

---
## RESULTS

In [ ]:
print("=" * 60)
print("PADDLEOCR COMPATIBILITY RESULTS")
print("=" * 60)
print(f"Method 1 (Latest + langchain 0.1.0):  {'WORKS' if METHOD_1 else 'FAILED'}")
print(f"Method 2 (PaddleOCR 2.8.1):           {'WORKS' if METHOD_2 else 'FAILED'}")
print(f"Method 3 (Bypass langchain):          {'WORKS' if METHOD_3 else 'FAILED'}")

print("\n" + "=" * 60)
if METHOD_1:
    print("USE: PaddleOCR 3.x with ocr.predict() API")
    print("API: result = ocr.predict(input='image.jpg')")
elif METHOD_2:
    print("USE: PaddleOCR 2.8.1 with ocr.ocr() API")
    print("API: result = ocr.ocr('image.jpg', cls=True)")
elif METHOD_3:
    print("USE: PaddleOCR 3.x with langchain bypass")
    print("API: result = ocr.predict(input='image.jpg')")
else:
    print("NO METHOD WORKED")
    print("Consider: Tesseract or Cloud APIs")

---
## Quick Test (if any method worked)

In [ ]:
# Upload image
if METHOD_1 or METHOD_2 or METHOD_3:
    from google.colab import files
    print("Upload your test image:")
    uploaded = files.upload()
    TEST_IMAGE = list(uploaded.keys())[0]
    print(f"Using: {TEST_IMAGE}")
else:
    print("No working method found.")

In [ ]:
# Run OCR
if METHOD_1 or METHOD_3:
    # PaddleOCR 3.x API
    result = ocr.predict(input=TEST_IMAGE)
    print("OCR Results:")
    for res in result:
        res.print()
        
elif METHOD_2:
    # PaddleOCR 2.x API
    result = ocr.ocr(TEST_IMAGE, cls=True)
    print("OCR Results:")
    if result[0]:
        for line in result[0]:
            text = line[1][0]
            conf = line[1][1]
            print(f"  {text} (conf: {conf:.2f})")